In [1]:
import matplotlib
matplotlib.use('Qt5Agg')  # Important for VSCode interactivity

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import PolygonSelector
from matplotlib.path import Path
from matplotlib.colors import LinearSegmentedColormap
import scanpy as sc

import numpy as np
import os
import plotly.io as pio
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import scipy.sparse as sp
from anndata import AnnData



In [2]:
class PolygonAnnotator:
    def __init__(self, image, cmap='hot'):
        self.image = image
        self.coords = None
        self.done = False

        self.fig, self.ax = plt.subplots()
        self.ax.imshow(image, cmap=cmap)
        self.ax.set_title("Click to draw polygon. Double-click to finish.")

        self.selector = PolygonSelector(
            self.ax,
            self.onselect,
            useblit=True,
            props=dict(color='cyan', linewidth=2),
            handle_props=dict(marker='o', markersize=5, mec='cyan', mfc='cyan')
        )

        self.fig.canvas.mpl_connect('close_event', self._on_close)
        plt.show(block=True)  # Blocks until the window is closed

    def _on_close(self, event):
        self.done = True

    def onselect(self, verts):
        self.coords = verts
        print(f"Polygon drawn with {len(verts)} points.")
        plt.close(self.fig)  # Automatically close the window

    def get_mask(self):
        if not self.coords:
            print("No polygon was drawn.")
            return None
        ny, nx = self.image.shape
        y, x = np.mgrid[:ny, :nx]
        points = np.vstack((x.flatten(), y.flatten())).T
        path = Path(self.coords)
        return path.contains_points(points).reshape((ny, nx))

In [3]:
def get_mz_heatmap_image(adata, target_mz, tol=0.1):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    x = adata.obs["x"].values.astype(int)
    y = adata.obs["y"].values.astype(int)

    width = x.max() + 1
    height = y.max() + 1
    image = np.zeros((height, width))
    image[y, x] = intensities

    return image, matched_mz

In [4]:
# Load and visualize the m/z heatmap from AnnData
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad"
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad


In [5]:
def sort_adata_by_mz(adata, mz_column="mzs"):
    def to_float_with_warning(index, label="var_names"):
        mzs = pd.to_numeric(index, errors="coerce")
        n_nans = mzs.isna().sum()
        if n_nans > 0:
            print(f"⚠️ Warning: {n_nans} entries in {label} could not be converted to float and were set to NaN.")
        return mzs
    adata.var[mz_column] = to_float_with_warning(adata.var_names)
    adata = adata[:, adata.var[mz_column].sort_values().index]
    return adata


In [6]:
adata = sort_adata_by_mz(adata, mz_column="mzs")
adata.var

,mzs
136.0150,136.0150
136.0608,136.0608
137.0251,137.0251
138.0287,138.0287
139.0359,139.0359
...,...
1520.1588,1520.1588
1521.1633,1521.1633
1522.1682,1522.1682
1542.1475,1542.1475


In [7]:
# ---- Run ----

target_mz = 788.5
image, matched_mz = get_mz_heatmap_image(adata, target_mz)
print(f"Target m/z: {target_mz} → Matched m/z: {matched_mz}")

'''custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])
'''
custom_cmap = "binary"
# Use the annotator with your image and colormap
annotator = PolygonAnnotator(image, cmap=custom_cmap)
mask = annotator.get_mask()

if mask is not None:
    masked_image = np.where(mask, image, 0)
    plt.imshow(masked_image, cmap=custom_cmap)
    plt.title(f"Masked m/z = {matched_mz:.4f}")
    plt.show()
else:
    print("No mask created.")

Target m/z: 788.5 → Matched m/z: 788.5027
Polygon drawn with 34 points.


In [8]:
# Get spatial coordinates
x = adata.obs["x"].astype(int).values
y = adata.obs["y"].astype(int).values

# Mask value at each spot
adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)
adata.obs["tissue"] = adata.obs["tissue"].astype(int)  # 1 = inside, 0 = outside
print(adata.obs["tissue"].value_counts())

tissue
0    9454
1    7151
Name: count, dtype: int64


/var/folders/0n/w0dydx896p7b5chql0_cv6hr0000gn/T/ipykernel_21693/1596079847.py:6: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)


In [9]:
adata1 = adata
adata.obs["tissue"]

0        0
1        0
2        0
3        0
4        0
        ..
16600    0
16601    0
16602    0
16603    0
16604    0
Name: tissue, Length: 16605, dtype: int64

In [10]:
adata = adata1
adata

AnnData object with n_obs × n_vars = 16605 × 1000
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue'
    var: 'mzs'

In [11]:
def filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=2.0, layer=None, normalize_tic=True):
    # 1. Extract raw data
    data = adata.layers[layer] if layer is not None else adata.X

    # 2. TIC normalization per pixel
    if normalize_tic:
        # Sum across features for each observation (axis=1)
        tic = np.asarray(data.sum(axis=1)).ravel()
        # Avoid division by zero
        tic[tic == 0] = 1.0
        # Normalize in place (if sparse, convert to array)
        data = data / tic[:, None]

    # 3. Define tissue masks
    mask_tissue = adata.obs[tissue_key] == tissue_val
    mask_non    = adata.obs[tissue_key] == non_tissue_val

    # 4. Compute mean normalized intensity per feature
    mean_tissue    = np.asarray(data[mask_tissue].mean(axis=0)).ravel()
    mean_nontissue = np.asarray(data[mask_non].mean(axis=0)).ravel()

    # 5. Identify features to remove
    to_remove = mean_tissue < foldchange * mean_nontissue

    # 6. Print removed m/z names
    mz_to_remove = adata.var_names[to_remove]
    print(f"Dropping {len(mz_to_remove)} m/z features:", mz_to_remove.values)

    # 7. Subset and return
    return adata[:, ~to_remove].copy()

In [12]:
foldchange = 3

# Filter m/z features based on tissue mask
#adata = filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=foldchange, layer=None, normalize_tic=True)

In [13]:
adata

AnnData object with n_obs × n_vars = 16605 × 1000
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue'
    var: 'mzs'

In [14]:
adata.var_names

Index(['136.0150', '136.0608', '137.0251', '138.0287', '139.0359', '154.0255',
       '155.0350', '156.0424', '159.0070', '160.0107',
       ...
       '1249.0300', '1255.0399', '1271.0072', '1494.1565', '1495.1523',
       '1520.1588', '1521.1633', '1522.1682', '1542.1475', '1544.1542'],
      dtype='object', length=1000)

In [15]:
def plot_mz_across_sample_custom_tic(adata, target_mz, tol=0.1):
    """
    Plots the TIC-normalized intensity of a specific m/z across all pixels using spatial coordinates.

    Parameters:
    - adata: AnnData object with MSI data
    - target_mz: float, the m/z value of interest
    - tol: float, tolerance for matching m/z (default ±0.1)
    """
    # Convert var_names to float (m/z axis)
    mz_axis = adata.var_names.astype(float).values

    # Find index of the m/z closest to the target within tolerance
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    # Extract intensities for that m/z from all pixels
    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]

    # Normalize by TIC
    if "TIC" not in adata.obs.columns:
        raise KeyError("TIC not found in adata.obs. Compute TIC before using this function.")
    tic = adata.obs["TIC"].values
    norm_intensities = intensities / tic
    norm_intensities[np.isnan(norm_intensities)] = 0  # Avoid NaNs from division by 0

    # Extract spatial coordinates
    if "x" not in adata.obs.columns or "y" not in adata.obs.columns:
        raise KeyError("Spatial coordinates 'x' and 'y' not found in adata.obs")

    x = adata.obs["x"].values
    y = adata.obs["y"].values

    # Create DataFrame for plotting
    df = pd.DataFrame({"x": x, "y": y, "intensity": norm_intensities})

    # Plot as heatmap using plotly express
    fig = px.scatter(
        df,
        x="x",
        y="y",
        color="intensity",
        color_continuous_scale=[
            [0.0, "#000000"],  # black
            [0.10, "#000080"],  # navy
            [0.15, "#0000FF"],  # blue
            [0.30, "#8000FF"],  # purple-ish
            [0.45, "#FF0000"],  # red
            [0.60, "#FF8000"],  # orange
            [0.75, "#FFFF00"],  # yellow
            [1.0, "#FFFFFF"]   # white
        ],
        title=f"TIC-Normalized Intensity Map of m/z = {matched_mz:.4f}",
        labels={"intensity": "Normalized Intensity"},
        height=500
    )

    fig.update_layout(yaxis=dict(scaleanchor="x", autorange="reversed"))
    return fig

In [16]:


# Ensure the image folder exists
os.makedirs(f"images_{foldchange}", exist_ok=True)

# Compute the global spectrum
global_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()

# Get indices of m/z values sorted by intensity (descending)
sorted_indices = np.argsort(global_spectrum)[::-1]

# Select indices from 10th to 20th (ranked #10 to #20)
selected_indices = sorted_indices[0:]

# Get corresponding m/z values
selected_mz = adata.var_names[selected_indices].astype(float)
"""
# Modified plotting loop
for mz in selected_mz:
    # Generate figure using your existing function
    fig = plot_mz_across_sample_custom_tic(adata, mz, tol=0.001)  # Modify function to return fig
    
    # Define filename
    filename = f"images_{foldchange}/mz_{mz:.4f}.png"
    
    # Save the figure
    pio.write_image(fig, filename)
    print(f"Saved: {filename}")

"""

'\n# Modified plotting loop\nfor mz in selected_mz:\n    # Generate figure using your existing function\n    fig = plot_mz_across_sample_custom_tic(adata, mz, tol=0.001)  # Modify function to return fig\n    \n    # Define filename\n    filename = f"images_{foldchange}/mz_{mz:.4f}.png"\n    \n    # Save the figure\n    pio.write_image(fig, filename)\n    print(f"Saved: {filename}")\n\n'

In [17]:
def mask_low_background_intensities(adata, fold_change=0.1):
    """
    Set background pixel intensities (tissue==0) to 0 if they are below fold_change * max tissue intensity per m/z.
    
    Parameters:
        adata : AnnData
            An AnnData object where .X is (n_pixels x n_mz) and .obs["tissue"] exists.
        fold_change : float
            The threshold multiplier to compare background intensity to max tissue intensity.
    """
    X = adata.X
    tissue_mask = adata.obs["tissue"] == 1
    background_mask = ~tissue_mask

    if sp.issparse(X):
        X = X.tocsr()

    # Compute max intensity of each m/z in tissue pixels
    max_tissue_intensity = X[tissue_mask.values].max(axis=0).A1 if sp.issparse(X) else X[tissue_mask.values].max(axis=0)

    # Define threshold per m/z
    thresholds = fold_change * max_tissue_intensity

    # Modify background pixels
    bg_indices = np.where(background_mask)[0]

    for i in bg_indices:
        row = X[i]
        if sp.issparse(X):
            row = row.tocoo()
            mask = row.data <= thresholds[row.col]
            row.data[mask] = 0
            X[i] = sp.csr_matrix((row.data, (np.zeros_like(row.col), row.col)), shape=(1, X.shape[1]))
        else:
            mask = row <= thresholds
            row[mask] = 0
            X[i] = row

    adata.X = X
    return adata

In [18]:
#adata = mask_low_background_intensities(adata, fold_change=0.01)

In [19]:
def export_pixel_areas_to_csv_with_centroid(
    adata: AnnData,
    pixel_area: float = 100.0,
    output_csv: str = "pixel_area_summary.csv",
    x_key: str = "x",
    y_key: str = "y",
):
    """
    Calculates and exports per-m/z area, intensity sum, and centroid info to a CSV.

    Parameters:
        adata : AnnData
            AnnData object with .X, .obs["tissue"], .obs["x"], .obs["y"], and .var["mzs"] or .var_names
        pixel_area : float
            Area of a single pixel (e.g., in μm²)
        output_csv : str
            Output CSV filename
        x_key : str
            Key in adata.obs for x coordinates
        y_key : str
            Key in adata.obs for y coordinates
    """
    X = adata.X
    if sp.issparse(X):
        X = X.tocsr()

    if "mzs" in adata.var.columns:
        mz_values = adata.var["mzs"].values
    else:
        mz_values = pd.to_numeric(adata.var_names, errors="coerce")
        print("Warning: 'mzs' not found in var. Using var_names instead.")

    # Masks
    tissue_mask = adata.obs["tissue"] == 1
    background_mask = ~tissue_mask

    x_coords = adata.obs[x_key].values
    y_coords = adata.obs[y_key].values

    # Initialize lists
    on_tissue_area = []
    non_tissue_area = []
    on_tissue_sum = []
    non_tissue_sum = []
    on_tissue_centroid_x = []
    on_tissue_centroid_y = []
    non_tissue_centroid_x = []
    non_tissue_centroid_y = []

    for mz_idx in range(X.shape[1]):
        intensities = X[:, mz_idx].toarray().flatten() if sp.issparse(X) else X[:, mz_idx]

        # Tissue and background
        tissue_values = intensities[tissue_mask.values]
        bg_values = intensities[background_mask.values]

        # Area
        on_tissue_area.append(np.sum(tissue_values > 0) * pixel_area)
        non_tissue_area.append(np.sum(bg_values > 0) * pixel_area)

        # Sum
        on_tissue_sum.append(np.sum(tissue_values))
        non_tissue_sum.append(np.sum(bg_values))

        # Centroid on tissue
        tx = x_coords[tissue_mask.values]
        ty = y_coords[tissue_mask.values]
        weight = tissue_values
        total = np.sum(weight)
        if total > 0:
            on_tissue_centroid_x.append(np.sum(tx * weight) / total)
            on_tissue_centroid_y.append(np.sum(ty * weight) / total)
        else:
            on_tissue_centroid_x.append(np.nan)
            on_tissue_centroid_y.append(np.nan)

        # Centroid on background
        bx = x_coords[background_mask.values]
        by = y_coords[background_mask.values]
        weight = bg_values
        total = np.sum(weight)
        if total > 0:
            non_tissue_centroid_x.append(np.sum(bx * weight) / total)
            non_tissue_centroid_y.append(np.sum(by * weight) / total)
        else:
            non_tissue_centroid_x.append(np.nan)
            non_tissue_centroid_y.append(np.nan)

    # DataFrame
    df = pd.DataFrame({
        "m/z": mz_values,
        "on_tissue_area": on_tissue_area,
        "non_tissue_area": non_tissue_area,
        "on_tissue_sum": on_tissue_sum,
        "non_tissue_sum": non_tissue_sum,
        "on_tissue_centroid_x": on_tissue_centroid_x,
        "on_tissue_centroid_y": on_tissue_centroid_y,
        "non_tissue_centroid_x": non_tissue_centroid_x,
        "non_tissue_centroid_y": non_tissue_centroid_y,
    })

    df.to_csv(output_csv, index=False)
    print(f"Saved to {output_csv}")
    return df

In [20]:
df = export_pixel_areas_to_csv_with_centroid(
    adata,
    pixel_area = 1.0,
    output_csv = "pixel_area_summary.csv",
    x_key = "x",
    y_key = "y",
)

Saved to pixel_area_summary.csv


In [21]:
df

,m/z,on_tissue_area,non_tissue_area,on_tissue_sum,non_tissue_sum,on_tissue_centroid_x,on_tissue_centroid_y,non_tissue_centroid_x,non_tissue_centroid_y
0,136.0150,6199.0,9440.0,507975.0,6863816.0,60.242973,50.807022,70.354641,75.642289
1,136.0608,7144.0,8590.0,2220296.0,1514359.0,61.953485,51.076435,66.858048,69.307963
2,137.0251,7151.0,9454.0,22357263.0,405797380.0,60.497221,52.789303,70.297145,85.396564
3,138.0287,7122.0,9454.0,2018837.0,31881184.0,60.681929,52.381017,70.173918,82.941998
4,139.0359,6636.0,9447.0,610775.0,6889345.0,61.628978,51.347067,70.584489,80.767940
...,...,...,...,...,...,...,...,...,...
995,1520.1588,7115.0,5406.0,688890.0,199117.0,62.044035,53.981562,73.849501,59.356514
996,1521.1633,7096.0,5300.0,613397.0,178975.0,62.111782,54.245192,73.823942,60.395776
997,1522.1682,7114.0,5395.0,663447.0,195938.0,62.082337,54.573931,72.907665,58.972226
998,1542.1475,7101.0,5290.0,543954.0,150146.0,61.943859,53.265522,71.585663,56.259268


In [22]:
import matplotlib.pyplot as plt
from skimage import measure

# Get contours at a constant value of 0.5 (between 0 and 1 in binary masks)
contours = measure.find_contours(mask, level=0.5)

# Plot the image with lower opacity and overlay the contour lines
plt.imshow(image, cmap='gray', alpha=0.3)  # Faded background
for contour in contours:
    plt.plot(contour[:, 1], contour[:, 0], linewidth=2, color='red')  # (x, y) = (col, row)

plt.axis('off')
plt.title('Polygon Margin Lines')
plt.show()

In [23]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from skimage import measure

def plot_mz_centroids_with_margin_and_mask_contours(
    df,
    image,
    mask,
    cmap_on_tissue="viridis",
    cmap_non_tissue="plasma",
    square_size=100,
    circle_size=50,
    figsize=(10, 8),
    show_margin=True,
    show_mask_contours=True,
    image_alpha=0.3,
    margin_linewidth=1.5,
    contour_color="red"
):
    """
    Plot centroid heatmap for each m/z using on-tissue squares and non-tissue circles.
    Optionally overlay the original image (faded) and polygon margin contours from mask.
    """
    required_cols = [
        "on_tissue_centroid_x", "on_tissue_centroid_y", "on_tissue_sum",
        "non_tissue_centroid_x", "non_tissue_centroid_y", "non_tissue_sum"
    ]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    norm_on_tissue = plt.Normalize(vmin=df["on_tissue_sum"].min(), vmax=df["on_tissue_sum"].max())
    norm_non_tissue = plt.Normalize(vmin=df["non_tissue_sum"].min(), vmax=df["non_tissue_sum"].max())

    cmap_on_tissue_obj = plt.colormaps[cmap_on_tissue]
    cmap_non_tissue_obj = plt.colormaps[cmap_non_tissue]

    fig, ax = plt.subplots(figsize=figsize)

    # Show background image with alpha
    ax.imshow(image, cmap='gray', alpha=image_alpha)

    # Plot centroids
    for _, row in df.iterrows():
        if not np.isnan(row["on_tissue_centroid_x"]) and not np.isnan(row["on_tissue_centroid_y"]):
            color = cmap_on_tissue_obj(norm_on_tissue(row["on_tissue_sum"]))
            square = patches.Rectangle(
                (row["on_tissue_centroid_x"] - square_size / 2, row["on_tissue_centroid_y"] - square_size / 2),
                square_size, square_size,
                linewidth=0.5, edgecolor="black", facecolor=color, alpha=0.8
            )
            ax.add_patch(square)

        if not np.isnan(row["non_tissue_centroid_x"]) and not np.isnan(row["non_tissue_centroid_y"]):
            color = cmap_non_tissue_obj(norm_non_tissue(row["non_tissue_sum"]))
            circle = patches.Circle(
                (row["non_tissue_centroid_x"], row["non_tissue_centroid_y"]),
                radius=circle_size / 2,
                linewidth=0.5, edgecolor="black", facecolor=color, alpha=0.8
            )
            ax.add_patch(circle)

    # Plot margin lines from mask
    if show_mask_contours and mask is not None:
        contours = measure.find_contours(mask, level=0.5)
        for contour in contours:
            ax.plot(contour[:, 1], contour[:, 0], linewidth=margin_linewidth, color=contour_color)

    # Optional dashed margin lines based on centroid averages
    if show_margin:
        tissue_centroids = df[["on_tissue_centroid_x", "on_tissue_centroid_y"]].dropna()
        non_tissue_centroids = df[["non_tissue_centroid_x", "non_tissue_centroid_y"]].dropna()

        if not tissue_centroids.empty:
            avg_tissue_x = tissue_centroids["on_tissue_centroid_x"].mean()
            avg_tissue_y = tissue_centroids["on_tissue_centroid_y"].mean()
            ax.plot([avg_tissue_x - 100, avg_tissue_x + 100],
                    [avg_tissue_y - 100, avg_tissue_y + 100],
                    color="black", linewidth=margin_linewidth, linestyle="--")

        if not non_tissue_centroids.empty:
            avg_non_tissue_x = non_tissue_centroids["non_tissue_centroid_x"].mean()
            avg_non_tissue_y = non_tissue_centroids["non_tissue_centroid_y"].mean()
            ax.plot([avg_non_tissue_x - 100, avg_non_tissue_x + 100],
                    [avg_non_tissue_y - 100, avg_non_tissue_y + 100],
                    color="red", linewidth=margin_linewidth, linestyle="--")

    # Colorbars
    sm_on_tissue = plt.cm.ScalarMappable(cmap=cmap_on_tissue_obj, norm=norm_on_tissue)
    sm_on_tissue.set_array([])
    plt.colorbar(sm_on_tissue, ax=ax, label="On-Tissue Intensity Sum")

    sm_non_tissue = plt.cm.ScalarMappable(cmap=cmap_non_tissue_obj, norm=norm_non_tissue)
    sm_non_tissue.set_array([])
    plt.colorbar(sm_non_tissue, ax=ax, label="Non-Tissue Intensity Sum")

    # Final plot settings
    ax.set_aspect('equal')
    ax.set_xlabel("X Coordinate (μm)")
    ax.set_ylabel("Y Coordinate (μm)")
    ax.set_title("m/z Centroids with Margin Lines and Image Background")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [24]:
plot_mz_centroids_with_margin_and_mask_contours(
    df,
    image,
    mask,
    cmap_on_tissue="viridis",
    cmap_non_tissue="plasma",
    square_size=0.5,
    circle_size=0.5,
    figsize=(6, 4),
    show_margin=False,
    show_mask_contours=True,
    image_alpha=0.3,
    margin_linewidth=1.5,
    contour_color="red"
)

In [25]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from skimage import measure

def plot_mz_centroids_with_margin_and_mask_contours(
    df,
    image,
    mask,
    cmap_on_tissue="viridis",
    cmap_non_tissue="plasma",
    square_size=100,
    circle_size=50,
    figsize=(10, 8),
    show_margin=False,
    show_mask_contours=True,
    image_alpha=0.3,
    margin_linewidth=1.5,
    contour_color="red"
):
    """
    Plot centroid heatmap for each m/z using on-tissue squares and non-tissue circles.
    Optionally overlay the original image (faded) and polygon margin contours from mask.
    Intensities are log-scaled before color mapping.
    """
    required_cols = [
        "on_tissue_centroid_x", "on_tissue_centroid_y", "on_tissue_sum",
        "non_tissue_centroid_x", "non_tissue_centroid_y", "non_tissue_sum"
    ]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    # Apply log1p to intensity values
    log_on_tissue = np.log1p(df["on_tissue_sum"])
    log_non_tissue = np.log1p(df["non_tissue_sum"])

    norm_on_tissue = plt.Normalize(vmin=log_on_tissue.min(), vmax=log_on_tissue.max())
    norm_non_tissue = plt.Normalize(vmin=log_non_tissue.min(), vmax=log_non_tissue.max())

    cmap_on_tissue_obj = plt.colormaps[cmap_on_tissue]
    cmap_non_tissue_obj = plt.colormaps[cmap_non_tissue]

    fig, ax = plt.subplots(figsize=figsize)

    # Show background image with alpha
    ax.imshow(image, cmap='gray', alpha=image_alpha)

    # Plot centroids
    for i, row in df.iterrows():
        if not np.isnan(row["on_tissue_centroid_x"]) and not np.isnan(row["on_tissue_centroid_y"]):
            color = cmap_on_tissue_obj(norm_on_tissue(log_on_tissue.iloc[i]))
            square = patches.Rectangle(
                (row["on_tissue_centroid_x"] - square_size / 2, row["on_tissue_centroid_y"] - square_size / 2),
                square_size, square_size,
                linewidth=0.5, edgecolor="black", facecolor=color, alpha=0.8
            )
            ax.add_patch(square)

        if not np.isnan(row["non_tissue_centroid_x"]) and not np.isnan(row["non_tissue_centroid_y"]):
            color = cmap_non_tissue_obj(norm_non_tissue(log_non_tissue.iloc[i]))
            circle = patches.Circle(
                (row["non_tissue_centroid_x"], row["non_tissue_centroid_y"]),
                radius=circle_size / 2,
                linewidth=0.5, edgecolor="black", facecolor=color, alpha=0.8
            )
            ax.add_patch(circle)

    # Plot mask contours
    if show_mask_contours and mask is not None:
        contours = measure.find_contours(mask, level=0.5)
        for contour in contours:
            ax.plot(contour[:, 1], contour[:, 0], linewidth=margin_linewidth, color=contour_color)

    # Colorbars
    sm_on_tissue = plt.cm.ScalarMappable(cmap=cmap_on_tissue_obj, norm=norm_on_tissue)
    sm_on_tissue.set_array([])
    plt.colorbar(sm_on_tissue, ax=ax, label="log(1 + On-Tissue Intensity Sum)")

    sm_non_tissue = plt.cm.ScalarMappable(cmap=cmap_non_tissue_obj, norm=norm_non_tissue)
    sm_non_tissue.set_array([])
    plt.colorbar(sm_non_tissue, ax=ax, label="log(1 + Non-Tissue Intensity Sum)")

    # Final plot settings
    ax.set_aspect('equal')
    ax.set_xlabel("X Coordinate (μm)")
    ax.set_ylabel("Y Coordinate (μm)")
    ax.set_title("m/z Centroids with Mask Contours and Log-Scaled Intensities")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [26]:
plot_mz_centroids_with_margin_and_mask_contours(
    df,
    image,
    mask,
    square_size=1,
    circle_size=1,
    figsize=(6, 4),
    show_margin=False,
    show_mask_contours=True,
    image_alpha=0.3,
    margin_linewidth=1.5,
    contour_color="red"
)

In [27]:
import numpy as np
import pandas as pd
from scipy.stats import entropy
from libpysal.weights import DistanceBand
from libpysal.weights import KNN
from esda.moran import Moran
from tqdm import tqdm

def compute_spatial_entropy(intensities):
    intensities = np.array(intensities, dtype=float)
    intensities = intensities[intensities > 0]
    if len(intensities) == 0:
        return 0
    p = intensities / intensities.sum()
    return entropy(p)


def compute_morans_I(x_coords, y_coords, intensities, k=8):
    coords = list(zip(x_coords, y_coords))
    try:
        w = KNN(coords, k=k)
        return Moran(intensities, w).I
    except Exception as e:
        print("Failed with error:", e)
        return np.nan

def analyze_scatter_wide_format(adata, k_neighbors=8):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values  # <-- Use "mzs"

    rows = []
    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):
        row = {"mz": mz_values[mz_index]}
        for flag, region in zip([1, 0], ["tissue", "background"]):
            subset = adata[adata.obs["tissue"] == flag]
            intensities = subset.X[:, mz_index].toarray().flatten() if hasattr(subset.X, "toarray") else subset.X[:, mz_index].flatten()
            x = subset.obs["x"].values
            y = subset.obs["y"].values

            row[f"spatial_entropy_{region}"] = compute_spatial_entropy(intensities)
            row[f"moran_I_{region}"] = compute_morans_I(x, y, intensities, k=k_neighbors)

        rows.append(row)

    return pd.DataFrame(rows)



In [28]:
"""
Moran's I and Spatial Entropy Analysis
This code computes Moran's I and spatial entropy for each m/z feature in the provided AnnData object.   
Moran's I is a measure of spatial autocorrelation, while spatial entropy quantifies the distribution of intensities. 
Moran's I values close to 1 indicate clustering, while values close to -1 indicate dispersion.
moran's I is computed using the `libpysal` library, which requires spatial coordinates and intensities.
Spatial entropy values range from 0 (uniform distribution) to log(n) (maximal entropy), where n is the number of pixels.   
spatial entropy is computed using the `scipy.stats.entropy` function, which requires the input to be a probability distribution.
The analysis is performed separately for tissue and background regions, and results are stored in a DataFrame.
The DataFrame contains the m/z values, spatial entropy, and Moran's I for both regions.
"""

"\nMoran's I and Spatial Entropy Analysis\nThis code computes Moran's I and spatial entropy for each m/z feature in the provided AnnData object.   \nMoran's I is a measure of spatial autocorrelation, while spatial entropy quantifies the distribution of intensities. \nMoran's I values close to 1 indicate clustering, while values close to -1 indicate dispersion.\nmoran's I is computed using the `libpysal` library, which requires spatial coordinates and intensities.\nSpatial entropy values range from 0 (uniform distribution) to log(n) (maximal entropy), where n is the number of pixels.   \nspatial entropy is computed using the `scipy.stats.entropy` function, which requires the input to be a probability distribution.\nThe analysis is performed separately for tissue and background regions, and results are stored in a DataFrame.\nThe DataFrame contains the m/z values, spatial entropy, and Moran's I for both regions.\n"

In [29]:
# Run it
scatter_wide_df = analyze_scatter_wide_format(adata, k_neighbors=8)

# Save to CSV (optional)
scatter_wide_df.to_csv("pixel_area_summary.csv", index=False)

# Preview
print(scatter_wide_df.head())

Processing m/z features: 100%|██████████| 1000/1000 [07:02<00:00,  2.37it/s]

         mz  spatial_entropy_tissue  moran_I_tissue  \
0  136.0150                8.402684        0.415658   
1  136.0608                8.702899        0.262929   
2  137.0251                8.616154        0.549667   
3  138.0287                8.596314        0.508409   
4  139.0359                8.537266        0.363029   

   spatial_entropy_background  moran_I_background  
0                    9.055275            0.493544  
1                    8.685133            0.494965  
2                    8.950049            0.721676  
3                    8.989311            0.701255  
4                    9.013560            0.650540  


In [30]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a DataFrame from the results
pixel_area_summary = pd.read_csv("pixel_area_summary.csv")

# Set up the figure size and layout
plt.figure(figsize=(8, 100))

# Plot heatmap for spatial entropy
sns.heatmap(scatter_wide_df[['spatial_entropy_tissue', 'spatial_entropy_background']].set_index(pixel_area_summary['mz']),
            cmap='coolwarm', annot=True, fmt=".2f", cbar=True, xticklabels=True, yticklabels=True)

plt.title("Spatial Entropy for Tissue vs. Background")
plt.xlabel("Region")
plt.ylabel("m/z")
plt.show()

In [31]:
# Plotting Spatial Entropy vs Moran's I for Tissue
plt.figure(figsize=(10, 6))

sns.scatterplot(data=pixel_area_summary, x="spatial_entropy_tissue", y="moran_I_tissue", hue="mz", palette="viridis")

plt.title("Spatial Entropy vs Moran's I for Tissue")
plt.xlabel("Spatial Entropy")
plt.ylabel("Moran's I")
plt.legend(title="m/z", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# Plotting Spatial Entropy vs Moran's I for Background
plt.figure(figsize=(10, 6))

sns.scatterplot(data=pixel_area_summary, x="spatial_entropy_background", y="moran_I_background", hue="mz", palette="viridis")

plt.title("Spatial Entropy vs Moran's I for Background")
plt.xlabel("Spatial Entropy")
plt.ylabel("Moran's I")
plt.legend(title="m/z", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [32]:
# Plotting Histograms of Spatial Entropy
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.histplot(pixel_area_summary["spatial_entropy_tissue"], bins=30, kde=True, color='blue', label='Tissue')
sns.histplot(pixel_area_summary["spatial_entropy_background"], bins=30, kde=True, color='red', label='Background')
plt.title("Histogram of Spatial Entropy")
plt.xlabel("Spatial Entropy")
plt.ylabel("Frequency")
plt.legend()

# Plotting Histograms of Moran's I
plt.subplot(1, 2, 2)
sns.histplot(pixel_area_summary["moran_I_tissue"].dropna(), bins=30, kde=True, color='blue', label='Tissue')
sns.histplot(pixel_area_summary["moran_I_background"].dropna(), bins=30, kde=True, color='red', label='Background')
plt.title("Histogram of Moran's I")
plt.xlabel("Moran's I")
plt.ylabel("Frequency")
plt.legend()

plt.tight_layout()
plt.show()

In [33]:
scatter_wide_df['moran_I_diff'] = scatter_wide_df['moran_I_tissue'] - scatter_wide_df['moran_I_background']
top_moran_I = scatter_wide_df.sort_values(by='moran_I_diff', ascending=False).head(10)

In [34]:
scatter_wide_df['entropy_diff'] = scatter_wide_df['spatial_entropy_background'] - scatter_wide_df['spatial_entropy_tissue']
top_entropy_diff = scatter_wide_df.sort_values(by='entropy_diff', ascending=False).head(10)

In [35]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(scatter_wide_df['spatial_entropy_tissue'], scatter_wide_df['moran_I_tissue'], label='Tissue', alpha=0.6)
plt.scatter(scatter_wide_df['spatial_entropy_background'], scatter_wide_df['moran_I_background'], label='Background', alpha=0.6, color='orange')
plt.xlabel("Spatial Entropy")
plt.ylabel("Moran's I")
plt.title("Spatial Statistics per m/z")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [36]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
plt.scatter(scatter_wide_df['spatial_entropy_tissue'], scatter_wide_df['moran_I_tissue'], label='Tissue', alpha=0.6)
plt.scatter(scatter_wide_df['spatial_entropy_background'], scatter_wide_df['moran_I_background'], label='Background', alpha=0.6, color='orange')

# Annotate each point with m/z value
for i, row in scatter_wide_df.iterrows():
    mz_label = f"{row['mz']:.1f}"  # format to 1 decimal
    plt.annotate(mz_label,
                 (row['spatial_entropy_tissue'], row['moran_I_tissue']),
                 textcoords="offset points",
                 xytext=(0, 4),
                 ha='center',
                 fontsize=6,
                 color='blue')
    plt.annotate(mz_label,
                 (row['spatial_entropy_background'], row['moran_I_background']),
                 textcoords="offset points",
                 xytext=(0, 4),
                 ha='center',
                 fontsize=6,
                 color='darkorange')

plt.xlabel("Spatial Entropy")
plt.ylabel("Moran's I")
plt.title("Spatial Statistics per m/z")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [37]:
import numpy as np
import pandas as pd
from tqdm import tqdm

def compute_intensity_metrics(adata):
    tissue_mask = adata.obs['tissue'] == 1
    background_mask = adata.obs['tissue'] == 0

    X = adata.X
    if hasattr(X, "toarray"):  # If sparse matrix
        X = X.toarray()

    mean_tissue = np.mean(X[tissue_mask], axis=0)
    mean_background = np.mean(X[background_mask], axis=0)

    total_tissue = np.sum(X[tissue_mask], axis=0)
    total_background = np.sum(X[background_mask], axis=0)

    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.divide(total_tissue, total_background)
        ratio[np.isnan(ratio)] = 0
        ratio[np.isinf(ratio)] = 0

    df = pd.DataFrame({
        "mz": adata.var["mzs"].values,
        "mean_intensity_tissue": mean_tissue,
        "mean_intensity_background": mean_background,
        "total_intensity_tissue": total_tissue,
        "total_intensity_background": total_background,
        "tissue_to_background_ratio": ratio
    })

    return df

In [38]:
intensity_df = compute_intensity_metrics(adata)


In [39]:
results_df = scatter_wide_df.merge(intensity_df, on="mz")


In [40]:
import matplotlib.pyplot as plt

def plot_intensity_vs_moransI(df):
    plt.figure(figsize=(10, 6))
    
    scatter = plt.scatter(
        df["tissue_to_background_ratio"],
        df["moran_I_tissue"],
        c=df["spatial_entropy_tissue"],
        s=np.clip(df["total_intensity_tissue"] / 1e5, 10, 200),  # scale size
        cmap="viridis",
        alpha=0.8,
        edgecolor='k',
        linewidth=0.5
    )

    plt.xlabel("Tissue-to-Background Intensity Ratio")
    plt.ylabel("Moran's I (Tissue)")
    plt.title("m/z Features: Spatial Autocorrelation vs. Intensity")
    cbar = plt.colorbar(scatter)
    cbar.set_label("Spatial Entropy (Tissue)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [41]:
plot_intensity_vs_moransI(results_df)


In [42]:
import matplotlib.pyplot as plt

def plot_intensity_vs_moransI_with_labels(df, top_n=10, score_fn=None):
    plt.figure(figsize=(10, 6))

    # Compute a custom score if not provided
    if score_fn is None:
        score_fn = lambda row: row["moran_I_tissue"] * row["tissue_to_background_ratio"] / (row["spatial_entropy_tissue"] + 1e-6)
    
    df = df.copy()
    df["score"] = df.apply(score_fn, axis=1)

    scatter = plt.scatter(
        df["tissue_to_background_ratio"],
        df["moran_I_tissue"],
        c=df["spatial_entropy_tissue"],
        s=np.clip(df["total_intensity_tissue"] / 1e5, 10, 200),
        cmap="viridis",
        alpha=0.8,
        edgecolor='k',
        linewidth=0.5
    )

    # Annotate top N points
    top_rows = df.nlargest(top_n, "score")
    for _, row in top_rows.iterrows():
        plt.annotate(
            f"{row['mz']:.1f}",
            (row["tissue_to_background_ratio"], row["moran_I_tissue"]),
            fontsize=9,
            xytext=(3, 3),
            textcoords='offset points'
        )

    plt.xlabel("Tissue-to-Background Intensity Ratio")
    plt.ylabel("Moran's I (Tissue)")
    plt.title(f"Top {top_n} m/z Features by Spatial-Intensity Score")
    cbar = plt.colorbar(scatter)
    cbar.set_label("Spatial Entropy (Tissue)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [43]:
"""
this function plots the relationship between tissue-to-background intensity ratio and Moran's I for m/z features.
It uses a scatter plot where the size of each point is proportional to the total intensity in tissue, and the color represents spatial entropy.
The function also annotates the top N m/z features based on a custom score, which is calculated as the product of Moran's I and the tissue-to-background ratio, normalized by spatial entropy.
The score is designed to highlight features that are both spatially autocorrelated and have a high intensity ratio.
The function allows for customization of the scoring function, enabling users to define their own criteria for selecting top features.  

"""

"\nthis function plots the relationship between tissue-to-background intensity ratio and Moran's I for m/z features.\nIt uses a scatter plot where the size of each point is proportional to the total intensity in tissue, and the color represents spatial entropy.\nThe function also annotates the top N m/z features based on a custom score, which is calculated as the product of Moran's I and the tissue-to-background ratio, normalized by spatial entropy.\nThe score is designed to highlight features that are both spatially autocorrelated and have a high intensity ratio.\nThe function allows for customization of the scoring function, enabling users to define their own criteria for selecting top features.  \n\n"

In [44]:
plot_intensity_vs_moransI_with_labels(results_df, top_n=50)


In [45]:
import numpy as np
import matplotlib.pyplot as plt

def plot_intensity_vs_moransI_with_labels(df, top_n=10, score_fn=None, background_threshold=1.0):
    plt.figure(figsize=(10, 6))

    if score_fn is None:
        score_fn = lambda row: row["moran_I_tissue"] * row["tissue_to_background_ratio"] / (row["spatial_entropy_tissue"] + 1e-6)

    df = df.copy()
    df["score"] = df.apply(score_fn, axis=1)

    # Define foreground and background based on threshold
    foreground_df = df[df["tissue_to_background_ratio"] >= background_threshold]
    background_df = df[df["tissue_to_background_ratio"] < background_threshold]

    # Plot background with triangle markers
    bg_scatter = plt.scatter(
        background_df["tissue_to_background_ratio"],
        background_df["moran_I_tissue"],
        c=background_df["spatial_entropy_tissue"],
        s=np.clip(background_df["total_intensity_tissue"] / 1e5, 10, 200),
        cmap="viridis",
        alpha=0.6,
        edgecolor='gray',
        linewidth=0.5,
        marker='v',  # triangle down
        label='Background m/z'
    )

    # Plot foreground with circle markers
    fg_scatter = plt.scatter(
        foreground_df["tissue_to_background_ratio"],
        foreground_df["moran_I_tissue"],
        c=foreground_df["spatial_entropy_tissue"],
        s=np.clip(foreground_df["total_intensity_tissue"] / 1e5, 10, 200),
        cmap="viridis",
        alpha=0.8,
        edgecolor='k',
        linewidth=0.5,
        marker='o',
        label='Foreground m/z'
    )

    # Annotate top N points from full dataframe
    top_rows = df.nlargest(top_n, "score")
    for _, row in top_rows.iterrows():
        plt.annotate(
            f"{row['mz']:.1f}",
            (row["tissue_to_background_ratio"], row["moran_I_tissue"]),
            fontsize=9,
            xytext=(3, 3),
            textcoords='offset points'
        )

    plt.xlabel("Tissue-to-Background Intensity Ratio")
    plt.ylabel("Moran's I (Tissue)")
    plt.title(f"Top {top_n} m/z Features by Spatial-Intensity Score")
    cbar = plt.colorbar(fg_scatter)
    cbar.set_label("Spatial Entropy (Tissue)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [46]:
plot_intensity_vs_moransI_with_labels(results_df, top_n=1000, background_threshold=1.0)


In [47]:
results_df

,mz,spatial_entropy_tissue,moran_I_tissue,spatial_entropy_background,moran_I_background,moran_I_diff,entropy_diff,mean_intensity_tissue,mean_intensity_background,total_intensity_tissue,total_intensity_background,tissue_to_background_ratio
0,136.0150,8.402684,0.415658,9.055275,0.493544,-0.077886,0.652591,71.035520,726.022424,507975.0,6863816.0,0.074008
1,136.0608,8.702899,0.262929,8.685133,0.494965,-0.232036,-0.017767,310.487484,160.181828,2220296.0,1514359.0,1.466162
2,137.0251,8.616154,0.549667,8.950049,0.721676,-0.172009,0.333895,3126.452664,42923.353078,22357263.0,405797380.0,0.055095
3,138.0287,8.596314,0.508409,8.989311,0.701255,-0.192846,0.392997,282.315341,3372.242860,2018837.0,31881184.0,0.063324
4,139.0359,8.537266,0.363029,9.013560,0.650540,-0.287511,0.476294,85.411131,728.722763,610775.0,6889345.0,0.088655
...,...,...,...,...,...,...,...,...,...,...,...,...
995,1520.1588,8.673163,0.286352,7.951131,0.600209,-0.313858,-0.722031,96.334778,21.061667,688890.0,199117.0,3.459725
996,1521.1633,8.660768,0.288768,7.961171,0.607453,-0.318685,-0.699598,85.777793,18.931140,613397.0,178975.0,3.427278
997,1522.1682,8.662518,0.290308,7.965635,0.599701,-0.309393,-0.696883,92.776814,20.725407,663447.0,195938.0,3.386005
998,1542.1475,8.677055,0.241838,8.104479,0.533510,-0.291672,-0.572576,76.066844,15.881743,543954.0,150146.0,3.622834


In [48]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def plot_top_mz_by_background_moransI(df, top_n=10, score_fn=None):
    plt.figure(figsize=(10, 6))

    # Default scoring function
    if score_fn is None:
        score_fn = lambda row: row["moran_I_tissue"] * row["tissue_to_background_ratio"] / (row["spatial_entropy_tissue"] + 1e-6)

    df = df.copy()
    df["score"] = df.apply(score_fn, axis=1)

    # Select top N rows by score
    top_rows = df.nlargest(top_n, "score")

    # Plot top N as circles
    scatter = plt.scatter(
        top_rows["moran_I_background"],
        top_rows["moran_I_tissue"],
        c=top_rows["spatial_entropy_tissue"],
        s=np.clip(top_rows["total_intensity_tissue"] / 1e5, 10, 200),
        cmap="viridis",
        alpha=0.9,
        edgecolor='black',
        linewidth=0.6,
        marker='o',
        label='Top m/z'
    )

    # Annotate m/z values
    for _, row in top_rows.iterrows():
        plt.annotate(
            f"{row['mz']:.1f}",
            (row["moran_I_background"], row["moran_I_tissue"]),
            fontsize=9,
            xytext=(3, 3),
            textcoords='offset points'
        )

    # Colorbar
    cbar = plt.colorbar(scatter)
    cbar.set_label("Spatial Entropy (Tissue)")

    # Size legend for circle areas (intensity)
    for val in [1e5, 3e5, 5e5]:
        plt.scatter([], [], s=val / 1e5, c='gray', alpha=0.5,
                    label=f"{int(val):,} total intensity")

    # Plot aesthetics
    plt.xlabel("Moran's I (Background)")
    plt.ylabel("Moran's I (Tissue)")
    plt.title(f"Top {top_n} m/z Features by Spatial-Intensity Score")
    plt.legend(title="Circle Size: Intensity", loc='lower right', frameon=True)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [49]:
plot_top_mz_by_background_moransI(results_df, top_n=1000,score_fn=None)

In [50]:

# Ensure the image folder exists
os.makedirs(f"images_{foldchange}", exist_ok=True)

# Compute the global spectrum
global_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()

# Get indices of m/z values sorted by intensity (descending)
sorted_indices = np.argsort(global_spectrum)[::-1]

# Select indices from 10th to 20th (ranked #10 to #20)
selected_indices = sorted_indices[0:]

# Get corresponding m/z values
selected_mz = adata.var_names[selected_indices].astype(float)

# Modified plotting loop
for mz in selected_mz:
    # Generate figure using your existing function
    fig = plot_mz_across_sample_custom_tic(adata, mz, tol=0.001)  # Modify function to return fig
    
    # Define filename
    filename = f"images_{foldchange}/mz_{mz:.4f}.png"
    
    # Save the figure
    pio.write_image(fig, filename)
    print(f"Saved: {filename}")


Saved: images_3/mz_273.0397.png
Saved: images_3/mz_137.0251.png
Saved: images_3/mz_274.0435.png
Saved: images_3/mz_369.3523.png
Saved: images_3/mz_331.0297.png
Saved: images_3/mz_257.0478.png
Saved: images_3/mz_195.0905.png
Saved: images_3/mz_637.3020.png
Saved: images_3/mz_313.0337.png
Saved: images_3/mz_275.0563.png
Saved: images_3/mz_245.0458.png
Saved: images_3/mz_333.0266.png
Saved: images_3/mz_197.9919.png
Saved: images_3/mz_244.0353.png
Saved: images_3/mz_246.0516.png
Saved: images_3/mz_734.5673.png
Saved: images_3/mz_330.0215.png
Saved: images_3/mz_347.0108.png
Saved: images_3/mz_551.0132.png
Saved: images_3/mz_258.0498.png
Saved: images_3/mz_229.0488.png
Saved: images_3/mz_501.0369.png
Saved: images_3/mz_199.0008.png
Saved: images_3/mz_332.0313.png
Saved: images_3/mz_544.9929.png
Saved: images_3/mz_798.5391.png
Saved: images_3/mz_375.0044.png
Saved: images_3/mz_290.0419.png
Saved: images_3/mz_370.3516.png
Saved: images_3/mz_291.0466.png
Saved: images_3/mz_638.3054.png
Saved: i

KeyboardInterrupt: 

In [52]:
features = results_df[[
    "moran_I_tissue",
    "moran_I_background",
    "spatial_entropy_tissue",
    "spatial_entropy_background"
]].dropna()  # Drop m/z features with missing values

In [53]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

In [54]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

# Add to original dataframe (after aligning indices)
results_df.loc[features.index, 'cluster'] = clusters

In [55]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
scatter = plt.scatter(
    results_df.loc[features.index, "moran_I_background"],
    results_df.loc[features.index, "moran_I_tissue"],
    c=results_df.loc[features.index, "cluster"],
    cmap="tab10",
    s=50,
    edgecolor='k'
)

plt.xlabel("Moran's I (Background)")
plt.ylabel("Moran's I (Tissue)")
plt.title("Clustering of m/z by Spatial Patterns")
plt.colorbar(scatter, label="Cluster ID")
plt.grid(True)
plt.tight_layout()
plt.show()

In [56]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Use the modern colormap accessor
from matplotlib import colormaps

# Create a colormap with N unique cluster colors
unique_clusters = results_df['cluster'].unique()
cmap = colormaps['tab10']  # or 'Set2', 'Dark2', etc.
cluster_colors = results_df.loc[features.index, 'cluster'].map(lambda x: cmap(x / len(unique_clusters))).to_numpy()

# Now plot the clustermap
sns.clustermap(X_scaled, cmap="viridis", row_colors=cluster_colors)
plt.show()


In [57]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# Create bin groupings based on Moran's I values
results_df["morans_group_tissue"] = (results_df["moran_I_tissue"] // 0.1).astype(int)
results_df["morans_group_background"] = (results_df["moran_I_background"] // 0.1).astype(int)

# Combine both into a single composite group for more refined clustering
results_df["morans_group_combo"] = results_df["morans_group_tissue"].astype(str) + "_" + results_df["morans_group_background"].astype(str)

# Prepare data for clustering
features = results_df[["moran_I_tissue", "moran_I_background", "spatial_entropy_tissue", "spatial_entropy_background"]]
X_scaled = StandardScaler().fit_transform(features)

# Assign a unique color to each Moran group
group_labels = results_df["morans_group_combo"].astype("category")
group_colors = group_labels.cat.codes
cmap = plt.colormaps['tab20']
row_colors = [cmap(code / group_colors.max()) for code in group_colors]

# Plot the clustermap
sns.clustermap(
    X_scaled,
    cmap="viridis",
    row_colors=np.array(row_colors),
    xticklabels=["Moran I (Tissue)", "Moran I (Background)", "Entropy (Tissue)", "Entropy (Background)"],
    figsize=(10, 10)
)
plt.show()

In [58]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def plot_top_mz_3d(df, top_n=10, score_fn=None):
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Default scoring function
    if score_fn is None:
        score_fn = lambda row: row["moran_I_tissue"] * row["tissue_to_background_ratio"] / (row["spatial_entropy_tissue"] + 1e-6)

    df = df.copy()
    df["score"] = df.apply(score_fn, axis=1)

    # Select top N rows by score
    top_rows = df.nlargest(top_n, "score")

    # Data to plot
    x = top_rows["moran_I_background"]
    y = top_rows["moran_I_tissue"]
    z = top_rows["tissue_to_background_ratio"]
    size = np.clip(top_rows["total_intensity_tissue"] / 1e5, 10, 200)
    color = top_rows["spatial_entropy_tissue"]

    # Scatter plot
    sc = ax.scatter(x, y, z, c=color, s=size, cmap='viridis', edgecolor='black', alpha=0.9)

    # Annotate m/z values
    for _, row in top_rows.iterrows():
        ax.text(row["moran_I_background"], row["moran_I_tissue"], row["tissue_to_background_ratio"],
                f"{row['mz']:.1f}", fontsize=9, ha='left', va='bottom')

    # Axes labels
    ax.set_xlabel("Moran's I (Background)")
    ax.set_ylabel("Moran's I (Tissue)")
    ax.set_zlabel("Tissue-to-Background Intensity Ratio")
    ax.set_title(f"Top {top_n} m/z Features (3D View)")

    # Colorbar
    cbar = fig.colorbar(sc, ax=ax, shrink=0.6)
    cbar.set_label("Spatial Entropy (Tissue)")

    plt.tight_layout()
    plt.show()

In [59]:
plot_top_mz_3d(results_df, top_n=1000,score_fn=None)

In [60]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def plot_top_mz_3d(df, top_n=10, score_fn=None):
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Default scoring function
    if score_fn is None:
        score_fn = lambda row: row["moran_I_tissue"] * row["tissue_to_background_ratio"] / (row["spatial_entropy_tissue"] + 1e-6)

    df = df.copy()
    df["score"] = df.apply(score_fn, axis=1)

    # Select top N rows by score
    top_rows = df.nlargest(top_n, "score")

    # Data to plot
    x = top_rows["moran_I_background"]
    y = top_rows["moran_I_tissue"]
    z = top_rows["tissue_to_background_ratio"]
    size = np.clip(top_rows["total_intensity_tissue"] / 1e5, 10, 200)
    color = top_rows["spatial_entropy_tissue"]

    # Scatter plot
    sc = ax.scatter(x, y, z, c=color, s=size, cmap='viridis', edgecolor='black', alpha=0.9)

    """# Annotate m/z values
    for _, row in top_rows.iterrows():
        ax.text(row["moran_I_background"], row["moran_I_tissue"], row["tissue_to_background_ratio"],
                f"{row['mz']:.1f}", fontsize=9, ha='left', va='bottom')
    """

    # Axes labels
    ax.set_xlabel("Moran's I (Background)")
    ax.set_ylabel("Moran's I (Tissue)")
    ax.set_zlabel("Tissue-to-Background Intensity Ratio")
    ax.set_title(f"Top {top_n} m/z Features (3D View)")

    # Colorbar
    cbar = fig.colorbar(sc, ax=ax, shrink=0.6)
    cbar.set_label("Spatial Entropy (Tissue)")

    plt.tight_layout()
    plt.show()

In [61]:
plot_top_mz_3d(results_df, top_n=1000,score_fn=None)

In [62]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def plot_top_mz_3d(df, top_n=10, score_fn=None):
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Default scoring function (note the swapped fields)
    if score_fn is None:
        score_fn = lambda row: row["moran_I_tissue"] * row["spatial_entropy_tissue"] / (row["tissue_to_background_ratio"] + 1e-6)

    df = df.copy()
    df["score"] = df.apply(score_fn, axis=1)

    # Select top N rows by score
    top_rows = df.nlargest(top_n, "score")

    # Data to plot
    x = top_rows["moran_I_background"]
    y = top_rows["moran_I_tissue"]
    z = top_rows["spatial_entropy_tissue"]  # Now Z-axis is spatial entropy
    size = np.clip(top_rows["total_intensity_tissue"] / 1e5, 10, 200)
    color = top_rows["tissue_to_background_ratio"]  # Now color reflects tissue/background ratio

    # Scatter plot
    sc = ax.scatter(x, y, z, c=color, s=size, cmap='viridis', edgecolor='black', alpha=0.9)

    # Annotate m/z values
    for _, row in top_rows.iterrows():
        ax.text(row["moran_I_background"], row["moran_I_tissue"], row["spatial_entropy_tissue"],
                f"{row['mz']:.1f}", fontsize=9, ha='left', va='bottom')

    # Axes labels
    ax.set_xlabel("Moran's I (Background)")
    ax.set_ylabel("Moran's I (Tissue)")
    ax.set_zlabel("Spatial Entropy (Tissue)")
    ax.set_title(f"Top {top_n} m/z Features (3D View)")

    # Colorbar
    cbar = fig.colorbar(sc, ax=ax, shrink=0.6)
    cbar.set_label("Tissue-to-Background Intensity Ratio")

    plt.tight_layout()
    plt.show()

In [63]:
plot_top_mz_3d(results_df, top_n=100,score_fn=None)

In [64]:
def plot_top_mz_3d(df, top_n=10, score_fn=None):
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Default scoring function (swapped spatial_entropy and tissue_to_background_ratio)
    if score_fn is None:
        score_fn = lambda row: row["moran_I_tissue"] * row["spatial_entropy_tissue"] / (row["tissue_to_background_ratio"] + 1e-6)

    df = df.copy()
    df["score"] = df.apply(score_fn, axis=1)

    # Select top N rows by score
    top_rows = df.nlargest(top_n, "score")

    # Data to plot
    x = top_rows["moran_I_background"]
    y = top_rows["moran_I_tissue"]
    z = top_rows["spatial_entropy_tissue"]  # Now Z-axis is spatial entropy
    size = np.clip(top_rows["total_intensity_tissue"] / 1e5, 10, 200)
    color = top_rows["tissue_to_background_ratio"]  # Now color reflects tissue/background ratio

    # Scatter plot (no labels)
    sc = ax.scatter(x, y, z, c=color, s=size, cmap='viridis', edgecolor='black', alpha=0.9)

    # Axes labels
    ax.set_xlabel("Moran's I (Background)")
    ax.set_ylabel("Moran's I (Tissue)")
    ax.set_zlabel("Spatial Entropy (Tissue)")
    ax.set_title(f"Top {top_n} m/z Features (3D View)")

    # Colorbar
    cbar = fig.colorbar(sc, ax=ax, shrink=0.6)
    cbar.set_label("Tissue-to-Background Intensity Ratio")

    plt.tight_layout()
    plt.show()

In [65]:
plot_top_mz_3d(results_df, top_n=1000,score_fn=None)

In [66]:
import numpy as np
import pandas as pd
from scipy.stats import skew
from sklearn.preprocessing import StandardScaler
import umap
import hdbscan
from skimage.measure import label, regionprops
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA

# Sample structure of the required data
# Assuming `results_df` has m/z metadata and intensity maps in a dict `intensity_images`
# Each `intensity_images[mz]` is a 2D numpy array of the ion image

# Placeholder: Simulated data structure
np.random.seed(42)
mz_values = np.linspace(100, 1000, 50)
intensity_images = {mz: np.random.rand(50, 50) for mz in mz_values}

# Metrics container
metrics = []

for mz, img in intensity_images.items():
    flat_img = img.flatten()
    
    # Basic intensity stats
    mean_intensity = np.mean(flat_img)
    std_intensity = np.std(flat_img)
    cv = std_intensity / mean_intensity if mean_intensity > 0 else 0
    skewness = skew(flat_img)
    
    # Spatial Entropy
    p = flat_img / flat_img.sum() if flat_img.sum() > 0 else np.ones_like(flat_img) / len(flat_img)
    spatial_entropy = -np.sum(p * np.log2(p + 1e-9))
    
    # Binarize image for region analysis
    binary_img = img > np.percentile(img, 90)
    labeled = label(binary_img)
    props = regionprops(labeled)
    region_area = np.sum([prop.area for prop in props])
    
    # Placeholder Moran's I and Geary's C
    # Normally this requires spatial weights; we'll mock with PCA-based proxy
    coords = np.indices(img.shape).reshape(2, -1).T
    pca = PCA(n_components=1)
    spatial_signal = pca.fit_transform(flat_img.reshape(-1, 1)).flatten()
    dists = pairwise_distances(coords)
    weights = 1 / (dists + 1e-5)
    weights[np.isinf(weights)] = 0
    np.fill_diagonal(weights, 0)

    mean_x = np.mean(spatial_signal)
    diff = spatial_signal - mean_x
    moran_numerator = np.sum(weights * np.outer(diff, diff))
    moran_denominator = np.sum(diff ** 2)
    moran_I = (len(spatial_signal) / np.sum(weights)) * (moran_numerator / moran_denominator)

    geary_numerator = np.sum(weights * (np.subtract.outer(spatial_signal, spatial_signal) ** 2))
    geary_denominator = 2 * np.sum(diff ** 2)
    geary_C = ((len(spatial_signal) - 1) / (2 * np.sum(weights))) * (geary_numerator / geary_denominator)

    metrics.append({
        "mz": mz,
        "mean_intensity": mean_intensity,
        "cv": cv,
        "skewness": skewness,
        "spatial_entropy": spatial_entropy,
        "region_area": region_area,
        "moran_I": moran_I,
        "geary_C": geary_C
    })

# Convert to DataFrame
metrics_df = pd.DataFrame(metrics)

# Normalize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(metrics_df.drop(columns=["mz"]))

# UMAP
umap_model = umap.UMAP(random_state=42)
X_umap = umap_model.fit_transform(X_scaled)

# HDBSCAN
clusterer = hdbscan.HDBSCAN(min_cluster_size=5)
clusters = clusterer.fit_predict(X_scaled)

# Add to DataFrame
metrics_df["cluster"] = clusters
metrics_df["umap1"] = X_umap[:, 0]
metrics_df["umap2"] = X_umap[:, 1]

metrics_df.head()

/opt/anaconda3/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



,mz,mean_intensity,cv,skewness,spatial_entropy,region_area,moran_I,geary_C,cluster,umap1,umap2
0,100.000000,0.498878,0.582962,-0.008044,11.003021,250.0,-0.001236,0.499982,-1,8.347518,2.806942
1,118.367347,0.494786,0.582803,0.015896,11.004135,250.0,-0.000290,0.498590,-1,7.883735,4.148122
2,136.734694,0.493635,0.582754,0.055338,11.007559,250.0,-0.002185,0.501918,-1,7.114465,2.311029
3,155.102041,0.489339,0.579304,0.035365,11.008217,250.0,-0.000859,0.498512,-1,7.581722,3.648052
4,173.469388,0.504624,0.576101,-0.006412,11.010138,250.0,-0.000792,0.498565,-1,6.350266,5.421074


In [67]:
adata.obs.head()

,x,y,TIC,sample,batch,age_group,disease_status,tissue
0,0,0,997479.0,4-3,Slide_3,Aged,AD,0
1,1,0,875098.0,4-3,Slide_3,Aged,AD,0
2,2,0,1357115.0,4-3,Slide_3,Aged,AD,0
3,3,0,1242398.0,4-3,Slide_3,Aged,AD,0
4,4,0,799114.0,4-3,Slide_3,Aged,AD,0


In [68]:
import numpy as np
import pandas as pd
from scipy.stats import entropy, variation, skew
from libpysal.weights import KNN
from esda.moran import Moran
from esda.geary import Geary
from tqdm import tqdm

def compute_spatial_entropy(intensities):
    intensities = np.array(intensities, dtype=float)
    intensities = intensities[intensities > 0]
    if len(intensities) == 0:
        return np.nan
    p = intensities / intensities.sum()
    return entropy(p)

def compute_morans_I(x_coords, y_coords, intensities, k=8):
    coords = list(zip(x_coords, y_coords))
    try:
        w = KNN(coords, k=k)
        return Moran(intensities, w).I
    except Exception as e:
        # Uncomment to debug
        # print(f"Moran's I calculation failed: {e}")
        return np.nan

def compute_gearys_C(x_coords, y_coords, intensities, k=8):
    coords = list(zip(x_coords, y_coords))
    try:
        w = KNN(coords, k=k)
        return Geary(intensities, w).C
    except Exception as e:
        # Uncomment to debug
        # print(f"Geary's C calculation failed: {e}")
        return np.nan

def analyze_metrics(adata, k_neighbors=8):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    rows = []
    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):
        row = {"mz": mz_values[mz_index]}

        for flag, region in zip([True, False], ["tissue", "background"]):
            subset = adata[adata.obs["tissue"] == flag]
            if subset.shape[0] == 0:
                # No spots for this region
                row[f"moran_I_{region}"] = np.nan
                row[f"geary_C_{region}"] = np.nan
                row[f"entropy_{region}"] = np.nan
                row[f"cv_{region}"] = np.nan
                row[f"skew_{region}"] = np.nan
                row[f"region_size"] = 0 if region == "tissue" else np.nan
                continue

            intensities = subset.X[:, mz_index].toarray().flatten() if hasattr(subset.X, "toarray") else subset.X[:, mz_index].flatten()
            x = subset.obs["x"].values
            y = subset.obs["y"].values

            # Filter out zero intensities for stats like CV and skew only if nonzero values exist
            nonzero_intensities = intensities[intensities > 0]

            row[f"moran_I_{region}"] = compute_morans_I(x, y, intensities, k=k_neighbors)
            row[f"geary_C_{region}"] = compute_gearys_C(x, y, intensities, k=k_neighbors)
            row[f"entropy_{region}"] = compute_spatial_entropy(intensities)

            if len(nonzero_intensities) > 0:
                row[f"cv_{region}"] = variation(nonzero_intensities)
                row[f"skew_{region}"] = skew(nonzero_intensities)
            else:
                row[f"cv_{region}"] = np.nan
                row[f"skew_{region}"] = np.nan

            if region == "tissue":
                row[f"region_size"] = np.sum(intensities > 0)

        rows.append(row)

    df_metrics = pd.DataFrame(rows)
    return df_metrics

In [69]:
# Run analysis
df_metrics = analyze_metrics(adata, k_neighbors=8)

# Drop rows with NaNs if needed (optional)
df_metrics_clean = df_metrics.dropna()

# Normalize & scale
from sklearn.preprocessing import StandardScaler
feature_cols = [col for col in df_metrics_clean.columns if col != "mz"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_metrics_clean[feature_cols])

# Dimensionality reduction
import umap
reducer = umap.UMAP(random_state=42)
X_umap = reducer.fit_transform(X_scaled)

# Clustering
import hdbscan
clusterer = hdbscan.HDBSCAN(min_cluster_size=5)
clusters = clusterer.fit_predict(X_scaled)

df_metrics_clean["cluster"] = clusters
df_metrics_clean["UMAP1"] = X_umap[:, 0]
df_metrics_clean["UMAP2"] = X_umap[:, 1]

print(df_metrics_clean.head())

Processing m/z features: 100%|██████████| 1000/1000 [24:39<00:00,  1.48s/it]
/opt/anaconda3/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



         mz  moran_I_tissue  geary_C_tissue  entropy_tissue  cv_tissue  \
0  136.0150        0.415658        0.547822        8.402684   0.965441   
1  136.0608        0.262929        0.729385        8.702899   0.630088   
2  137.0251        0.549667        0.406121        8.616154   0.964382   
3  138.0287        0.508409        0.452354        8.596314   0.921969   
4  139.0359        0.363029        0.612376        8.537266   0.820836   

   skew_tissue  region_size  moran_I_background  geary_C_background  \
0     3.751760         6199            0.493544            0.501015   
1     2.034254         7144            0.494965            0.504816   
2     7.986627         7151            0.721676            0.275891   
3     5.827630         7122            0.701255            0.295633   
4     3.310123         6636            0.650540            0.345992   

   entropy_background  cv_background  skew_background  cluster     UMAP1  \
0            9.055275       0.428758         0.29877

In [70]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
palette = sns.color_palette("hsv", len(df_metrics_clean["cluster"].unique()))
sns.scatterplot(
    data=df_metrics_clean, 
    x="UMAP1", y="UMAP2", 
    hue="cluster", 
    palette=palette,
    legend='full',
    s=60,
    edgecolor="k"
)
plt.title("UMAP Projection of m/z Feature Metrics")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [71]:
# Optional: filter out noise cluster (-1)
df_clusters_only = df_metrics_clean[df_metrics_clean["cluster"] != -1]

# Group and compute mean
cluster_means = df_clusters_only.groupby("cluster")[
    [col for col in df_clusters_only.columns if col not in ["mz", "UMAP1", "UMAP2", "cluster"]]
].mean()

# Normalize for better heatmap contrast (optional)
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

plt.figure(figsize=(12, 6))
sns.heatmap(cluster_means_norm, cmap="viridis", annot=True, fmt=".2f")
plt.title("Normalized Mean Metrics per Cluster")
plt.xlabel("Feature")
plt.ylabel("Cluster")
plt.tight_layout()
plt.show()

In [72]:
# For each cluster, print top m/z with highest entropy or other metric
for cluster_id in sorted(df_clusters_only["cluster"].unique()):
    top_mz = df_clusters_only[df_clusters_only["cluster"] == cluster_id].sort_values("entropy_tissue", ascending=False).head(3)
    print(f"Cluster {cluster_id} top m/z by entropy_tissue:")
    print(top_mz[["mz", "entropy_tissue"]])
    print()

Cluster 0 top m/z by entropy_tissue:
           mz  entropy_tissue
125  269.0502        8.476191
203  332.0313        8.469434
210  335.0136        8.445218

Cluster 1 top m/z by entropy_tissue:
           mz  entropy_tissue
433  489.1071        8.489396
520  564.9677        8.455395
307  387.1928        8.424431

Cluster 2 top m/z by entropy_tissue:
           mz  entropy_tissue
786  789.6188        8.654930
572  605.5548        8.649228
784  788.6130        8.648857

Cluster 3 top m/z by entropy_tissue:
           mz  entropy_tissue
82   240.0953        8.824960
64   224.1000        8.818300
788  790.5270        8.813275



In [73]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

# Select only numeric feature columns (exclude mz, cluster, UMAP1/2)
feature_cols = [
    col for col in df_metrics_clean.columns 
    if col not in ["mz", "cluster", "UMAP1", "UMAP2"]
]

# Standardize features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_metrics_clean[feature_cols])

# t-SNE embedding
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

# Add t-SNE results to DataFrame
df_metrics_clean["TSNE1"] = X_tsne[:, 0]
df_metrics_clean["TSNE2"] = X_tsne[:, 1]

# Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_metrics_clean,
    x="TSNE1", y="TSNE2",
    hue="cluster",
    palette="tab10",
    s=60,
    edgecolor="k",
    legend="full"
)
plt.title("t-SNE Projection of m/z Feature Metrics")
plt.xlabel("TSNE1")
plt.ylabel("TSNE2")
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

/opt/anaconda3/lib/python3.12/site-packages/sklearn/manifold/_t_sne.py:1162: FutureWarning:

'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.



In [74]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import umap
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# Step 1: Filter numeric features
feature_cols = [
    col for col in df_metrics_clean.columns 
    if col not in ["mz", "cluster", "UMAP1", "UMAP2", "TSNE1", "TSNE2"]
]

# Step 2: Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_metrics_clean[feature_cols])

# Step 3: Run t-SNE with tuning
tsne = TSNE(n_components=2, random_state=42, perplexity=40, learning_rate=200, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)
df_metrics_clean["TSNE1"] = X_tsne[:, 0]
df_metrics_clean["TSNE2"] = X_tsne[:, 1]

# Step 4: Run UMAP again to ensure consistent dimensions and comparison
umap_model = umap.UMAP(n_components=2, random_state=42)
X_umap = umap_model.fit_transform(X_scaled)
df_metrics_clean["UMAP1"] = X_umap[:, 0]
df_metrics_clean["UMAP2"] = X_umap[:, 1]

# Step 5: Plot with Plotly - Side-by-side interactive scatter plots
fig = make_subplots(rows=1, cols=2, subplot_titles=("t-SNE", "UMAP"), shared_yaxes=True)

# t-SNE plot
fig.add_trace(
    go.Scattergl(
        x=df_metrics_clean["TSNE1"],
        y=df_metrics_clean["TSNE2"],
        mode='markers',
        marker=dict(color=df_metrics_clean["cluster"], colorscale='Viridis', showscale=True),
        text=["m/z: {:.4f}".format(mz) for mz in df_metrics_clean["mz"]],
        name="t-SNE"
    ),
    row=1, col=1
)

# UMAP plot
fig.add_trace(
    go.Scattergl(
        x=df_metrics_clean["UMAP1"],
        y=df_metrics_clean["UMAP2"],
        mode='markers',
        marker=dict(color=df_metrics_clean["cluster"], colorscale='Viridis', showscale=False),
        text=["m/z: {:.4f}".format(mz) for mz in df_metrics_clean["mz"]],
        name="UMAP"
    ),
    row=1, col=2
)

fig.update_layout(
    title_text="Comparison of t-SNE and UMAP Projections",
    height=600,
    width=1000,
    showlegend=False
)

fig.update_xaxes(title_text="TSNE1", row=1, col=1)
fig.update_yaxes(title_text="TSNE2", row=1, col=1)
fig.update_xaxes(title_text="UMAP1", row=1, col=2)
fig.update_yaxes(title_text="UMAP2", row=1, col=2)

fig.show()


/opt/anaconda3/lib/python3.12/site-packages/sklearn/manifold/_t_sne.py:1162: FutureWarning:

'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.

/opt/anaconda3/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [75]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import umap
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Step 1: Prepare data
df = df_metrics.copy()
feature_cols = [col for col in df.columns if col not in ["mz", "cluster", "UMAP1", "UMAP2", "TSNE1", "TSNE2"]]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])

# Step 2: Run dimensionality reductions
tsne_1 = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42)
X_tsne_1 = tsne_1.fit_transform(X_scaled)

tsne_2 = TSNE(n_components=2, perplexity=50, learning_rate=50, random_state=42)
X_tsne_2 = tsne_2.fit_transform(X_scaled)

X_umap = umap.UMAP(n_components=2, random_state=42).fit_transform(X_scaled)

# Add to DataFrame
df["TSNE1_a"], df["TSNE2_a"] = X_tsne_1[:, 0], X_tsne_1[:, 1]
df["TSNE1_b"], df["TSNE2_b"] = X_tsne_2[:, 0], X_tsne_2[:, 1]
df["UMAP1"], df["UMAP2"] = X_umap[:, 0], X_umap[:, 1]

# Step 3: Interactive Plotly comparison
fig = make_subplots(rows=1, cols=2, subplot_titles=("t-SNE", "UMAP"), shared_yaxes=True)
color_column = "mz"
# t-SNE trace (initial = model 1)
tsne_trace = go.Scattergl(
    x=df["TSNE1_a"],
    y=df["TSNE2_a"],
    mode='markers',
    marker=dict(color=df[color_column], colorscale="Viridis", size=6, showscale=True),
    text=df["mz"],
    name="t-SNE (p=30, lr=200)"
)

# UMAP trace
umap_trace = go.Scattergl(
    x=df["UMAP1"],
    y=df["UMAP2"],
    mode='markers',
    marker=dict(color=df[color_column], colorscale="Viridis", size=6, showscale=False),
    text=df["mz"],
    name="UMAP"
)

# Add traces
fig.add_trace(tsne_trace, row=1, col=1)
fig.add_trace(umap_trace, row=1, col=2)

# Dropdown for switching t-SNE parameters
fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="down",
            buttons=[
                dict(
                    label="t-SNE (p=30, lr=200)",
                    method="restyle",
                    args=[{"x": [df["TSNE1_a"]], "y": [df["TSNE2_a"]]}, [0]]
                ),
                dict(
                    label="t-SNE (p=50, lr=50)",
                    method="restyle",
                    args=[{"x": [df["TSNE1_b"]], "y": [df["TSNE2_b"]]}, [0]]
                ),
            ],
            x=0.05, y=1.2,
            showactive=True
        )
    ]
)

# Layout adjustments
fig.update_layout(
    title="Interactive Comparison of t-SNE and UMAP",
    width=1000,
    height=500
)

fig.show()


/opt/anaconda3/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [76]:
from sklearn.cluster import MiniBatchKMeans

# Choose number of clusters
n_clusters = 100  # or any number between 100-200

# Run clustering
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=100)
df_metrics_clean["cluster"] = kmeans.fit_predict(X_scaled)

In [77]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, subplot_titles=("t-SNE", "UMAP"), shared_yaxes=True)

# t-SNE trace
fig.add_trace(
    go.Scattergl(
        x=df_metrics_clean["TSNE1"],
        y=df_metrics_clean["TSNE2"],
        mode="markers",
        marker=dict(
            color=df_metrics_clean["cluster"],
            colorscale="Viridis",
            size=5,
            showscale=True,
            colorbar=dict(title="Cluster")
        ),
        text=df_metrics_clean["mz"],
        name="t-SNE"
    ),
    row=1, col=1
)

# UMAP trace
fig.add_trace(
    go.Scattergl(
        x=df_metrics_clean["UMAP1"],
        y=df_metrics_clean["UMAP2"],
        mode="markers",
        marker=dict(
            color=df_metrics_clean["cluster"],
            colorscale="Viridis",
            size=5,
            showscale=False
        ),
        text=df_metrics_clean["mz"],
        name="UMAP"
    ),
    row=1, col=2
)

fig.update_layout(title="t-SNE vs UMAP with 150 Clusters", height=600, width=1000)
fig.show()

In [78]:
for cluster_id, group in df_metrics_clean.groupby("cluster"):
    print(f"\nCluster {cluster_id} ({len(group)} m/z features):")
    print(group["mz"].values)


Cluster 0 (14 m/z features):
[229.0488 244.0353 245.0458 246.0516 247.0594 256.0338 257.0478 262.0491
 272.0306 275.0563 289.0389 290.0419 291.0466 292.0531]

Cluster 1 (13 m/z features):
[163.0405 202.1793 203.0369 203.068  214.0577 233.0507 265.0616 275.2736
 342.3255 361.0668 455.0366 457.0507 609.0626]

Cluster 2 (12 m/z features):
[772.5231 773.5307 798.5391 799.5511 800.5515 801.5524 824.5594 846.5322
 848.5533 849.5585 912.4556 966.9248]

Cluster 3 (12 m/z features):
[ 841.0405  859.0277  876.0159  880.0137  882.0225  883.0213  899.0129
  905.032  1033.0245 1051.0231 1072.0057 1255.0399]

Cluster 4 (4 m/z features):
[230.9492 252.9273 268.8998 422.932 ]

Cluster 5 (11 m/z features):
[197.1082 349.039  350.0839 386.9952 407.1967 499.0256 580.9368 786.4932
 837.6503 858.5287 966.5311]

Cluster 6 (15 m/z features):
[364.0681 387.1928 389.9643 392.0363 465.1795 473.1425 537.9825 560.0797
 587.9561 588.9623 737.9794 740.9714 743.9934 744.9822 934.9747]

Cluster 7 (1 m/z features):
[

In [79]:
clustered_mzs = df_metrics_clean.groupby("cluster")["mz"].apply(list).to_dict()


In [80]:
clustered_mzs[0]  # m/z values in cluster 0
clustered_mzs[1]  # m/z values in cluster 1


[163.0405,
 202.1793,
 203.0369,
 203.068,
 214.0577,
 233.0507,
 265.0616,
 275.2736,
 342.3255,
 361.0668,
 455.0366,
 457.0507,
 609.0626]

In [81]:
df_metrics_clean[["mz", "cluster"]].sort_values("cluster").to_csv("clustered_mzs.csv", index=False)



In [82]:
cluster_counts = df_metrics_clean["cluster"].value_counts().sort_index()
print(cluster_counts)


cluster
0     14
1     13
2     12
3     12
4      4
      ..
95     7
96     4
97     5
98    20
99     2
Name: count, Length: 99, dtype: int64


In [83]:
import os
import plotly.io as pio

# Ensure the image folder exists
os.makedirs(f"images_from_clusters", exist_ok=True)

# Merge cluster info with AnnData's m/z values
mz_cluster_map = dict(zip(df_metrics_clean["mz"].astype(float), df_metrics_clean["cluster"]))

# Compute the global spectrum
global_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()

# Get indices of m/z values sorted by intensity (descending)
sorted_indices = np.argsort(global_spectrum)[::-1]

# Select indices of interest (e.g., all, top N, etc.)
selected_indices = sorted_indices[0:]  # You can limit with [10:20] if desired

# Get corresponding m/z values from adata.var
selected_mz = adata.var["mzs"].values[selected_indices].astype(float)

# Generate and save images
for mz in selected_mz:
    if mz not in mz_cluster_map:
        print(f"Skipping m/z {mz:.4f} — not in clustered data.")
        continue

    cluster_id = mz_cluster_map[mz]
    fig = plot_mz_across_sample_custom_tic(adata, mz, tol=0.001)  # Make sure this function returns a Plotly fig

    filename = f"images_from_clusters/cluster_{cluster_id}_mz_{mz:.4f}_.png"
    pio.write_image(fig, filename)
    print(f"Saved: {filename}")

Saved: images_from_clusters/cluster_76_mz_273.0397_.png
Saved: images_from_clusters/cluster_76_mz_137.0251_.png
Saved: images_from_clusters/cluster_76_mz_274.0435_.png
Saved: images_from_clusters/cluster_59_mz_369.3523_.png
Saved: images_from_clusters/cluster_81_mz_331.0297_.png
Saved: images_from_clusters/cluster_0_mz_257.0478_.png
Saved: images_from_clusters/cluster_28_mz_195.0905_.png
Saved: images_from_clusters/cluster_81_mz_637.3020_.png
Saved: images_from_clusters/cluster_76_mz_313.0337_.png
Saved: images_from_clusters/cluster_0_mz_275.0563_.png
Saved: images_from_clusters/cluster_0_mz_245.0458_.png
Saved: images_from_clusters/cluster_81_mz_333.0266_.png
Saved: images_from_clusters/cluster_81_mz_197.9919_.png
Saved: images_from_clusters/cluster_0_mz_244.0353_.png
Saved: images_from_clusters/cluster_0_mz_246.0516_.png
Saved: images_from_clusters/cluster_60_mz_734.5673_.png
Saved: images_from_clusters/cluster_81_mz_330.0215_.png
Saved: images_from_clusters/cluster_54_mz_347.0108_.p

KeyboardInterrupt: 

In [84]:
import numpy as np
import pandas as pd
from scipy.stats import entropy, variation, skew
from libpysal.weights import KNN
from esda.moran import Moran
from esda.geary import Geary
from tqdm import tqdm

# Define global offsets for common adducts and isotopes
ADDUCTS_AND_ISOTOPES = {
    "has_H_adduct": 1.0073,
    "has_Na_adduct": 22.9898,
    "has_K_adduct": 38.9637,
    "has_Mplus1_isotope": 1.00335,
    "has_Mplus2_isotope": 2.0067,
}

def find_related_mz_features(mz_values, current_mz, tolerance=0.01):
    related_features = {}
    for label, offset in ADDUCTS_AND_ISOTOPES.items():
        target_mz = current_mz + offset
        match = np.any(np.isclose(mz_values, target_mz, atol=tolerance))
        related_features[label] = int(match)
    return related_features

def compute_spatial_entropy(intensities):
    intensities = np.array(intensities, dtype=float)
    intensities = intensities[intensities > 0]
    if len(intensities) == 0:
        return np.nan
    p = intensities / intensities.sum()
    return entropy(p)

def compute_morans_I(x_coords, y_coords, intensities, k=8):
    coords = list(zip(x_coords, y_coords))
    try:
        w = KNN(coords, k=k)
        return Moran(intensities, w).I
    except Exception as e:
        # Uncomment to debug
        # print(f"Moran's I calculation failed: {e}")
        return np.nan

def compute_gearys_C(x_coords, y_coords, intensities, k=8):
    coords = list(zip(x_coords, y_coords))
    try:
        w = KNN(coords, k=k)
        return Geary(intensities, w).C
    except Exception as e:
        # Uncomment to debug
        # print(f"Geary's C calculation failed: {e}")
        return np.nan

def analyze_metrics(adata, k_neighbors=8):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    rows = []
    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):
        row = {"mz": mz_values[mz_index]}

        for flag, region in zip([True, False], ["tissue", "background"]):
            subset = adata[adata.obs["tissue"] == flag]
            if subset.shape[0] == 0:
                # No spots for this region
                row[f"moran_I_{region}"] = np.nan
                row[f"geary_C_{region}"] = np.nan
                row[f"entropy_{region}"] = np.nan
                row[f"cv_{region}"] = np.nan
                row[f"skew_{region}"] = np.nan
                row[f"region_size"] = 0 if region == "tissue" else np.nan
                continue

            intensities = subset.X[:, mz_index].toarray().flatten() if hasattr(subset.X, "toarray") else subset.X[:, mz_index].flatten()
            x = subset.obs["x"].values
            y = subset.obs["y"].values

            # Filter out zero intensities for stats like CV and skew only if nonzero values exist
            nonzero_intensities = intensities[intensities > 0]

            row[f"moran_I_{region}"] = compute_morans_I(x, y, intensities, k=k_neighbors)
            row[f"geary_C_{region}"] = compute_gearys_C(x, y, intensities, k=k_neighbors)
            row[f"entropy_{region}"] = compute_spatial_entropy(intensities)

            if len(nonzero_intensities) > 0:
                row[f"cv_{region}"] = variation(nonzero_intensities)
                row[f"skew_{region}"] = skew(nonzero_intensities)
            else:
                row[f"cv_{region}"] = np.nan
                row[f"skew_{region}"] = np.nan

            if region == "tissue":
                row[f"region_size"] = np.sum(intensities > 0)

        related_features = find_related_mz_features(mz_values, mz_values[mz_index])
        row.update(related_features)

        rows.append(row)

    df_metrics = pd.DataFrame(rows)
    return df_metrics

In [86]:

# Run analysis
df_metrics = analyze_metrics(adata, k_neighbors=8)

# Drop rows with NaNs if needed (optional)
df_metrics_clean = df_metrics.dropna()

# Normalize & scale
from sklearn.preprocessing import StandardScaler
feature_cols = [col for col in df_metrics_clean.columns if col != "mz"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_metrics_clean[feature_cols])

# UMAP dimensionality reduction
import umap
reducer = umap.UMAP(random_state=42)
X_umap = reducer.fit_transform(X_scaled)

# t-SNE dimensionality reduction
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

# Clustering
import hdbscan
clusterer = hdbscan.HDBSCAN(min_cluster_size=5)
clusters = clusterer.fit_predict(X_scaled)

# Add to DataFrame
df_metrics_clean["cluster"] = clusters
df_metrics_clean["UMAP1"] = X_umap[:, 0]
df_metrics_clean["UMAP2"] = X_umap[:, 1]
df_metrics_clean["tSNE1"] = X_tsne[:, 0]
df_metrics_clean["tSNE2"] = X_tsne[:, 1]

print(df_metrics_clean.head())

Processing m/z features: 100%|██████████| 1000/1000 [23:43<00:00,  1.42s/it]
/opt/anaconda3/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/opt/anaconda3/lib/python3.12/site-packages/sklearn/manifold/_t_sne.py:1162: FutureWarning:

'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.



         mz  moran_I_tissue  geary_C_tissue  entropy_tissue  cv_tissue  \
0  136.0150        0.415658        0.547822        8.402684   0.965441   
1  136.0608        0.262929        0.729385        8.702899   0.630088   
2  137.0251        0.549667        0.406121        8.616154   0.964382   
3  138.0287        0.508409        0.452354        8.596314   0.921969   
4  139.0359        0.363029        0.612376        8.537266   0.820836   

   skew_tissue  region_size  moran_I_background  geary_C_background  \
0     3.751760         6199            0.493544            0.501015   
1     2.034254         7144            0.494965            0.504816   
2     7.986627         7151            0.721676            0.275891   
3     5.827630         7122            0.701255            0.295633   
4     3.310123         6636            0.650540            0.345992   

   entropy_background  ...  has_H_adduct  has_Na_adduct  has_K_adduct  \
0            9.055275  ...             1              1

In [87]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set(style="whitegrid", rc={"figure.figsize": (10, 8)})

# UMAP plot
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x="UMAP1", y="UMAP2",
    hue="cluster", palette="tab10",
    data=df_metrics_clean, s=50, linewidth=0, legend='full'
)
plt.title("UMAP Clustering of m/z Features")
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# t-SNE plot
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x="tSNE1", y="tSNE2",
    hue="cluster", palette="tab10",
    data=df_metrics_clean, s=50, linewidth=0, legend='full'
)
plt.title("t-SNE Clustering of m/z Features")
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [88]:
from sklearn.cluster import MiniBatchKMeans

# Choose number of clusters
n_clusters = 100  # or any number between 100-200

# Run clustering
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=100)
df_metrics_clean["cluster"] = kmeans.fit_predict(X_scaled)

In [89]:
for cluster_id, group in df_metrics_clean.groupby("cluster"):
    print(f"\nCluster {cluster_id} ({len(group)} m/z features):")
    print(group["mz"].values)


Cluster 0 (9 m/z features):
[217.0514 218.056  261.0393 306.0433 397.0578 410.0475 425.0628 439.057
 603.5365]

Cluster 1 (21 m/z features):
[160.0107 318.0517 339.0374 366.0285 485.358  491.0306 513.0214 537.0208
 540.4258 553.9998 574.9745 631.0347 647.0388 667.0381 689.0372 711.0204
 713.001  729.0159 749.0038 775.098  791.0672]

Cluster 2 (6 m/z features):
[ 296.0663  497.3396  526.299   530.3032  534.2923 1495.1523]

Cluster 3 (21 m/z features):
[172.054  178.0269 184.072  197.1082 202.0613 202.1793 203.0369 215.0298
 240.0412 267.0272 305.0435 311.013  391.0428 594.0808 598.057  604.0622
 713.4442 724.5195 804.4848 804.5468 896.488 ]

Cluster 4 (14 m/z features):
[ 443.901   599.9377  821.5281  827.5714  849.5585  853.4827  869.4633
  872.556   875.4991  912.4556  963.4792  974.5565 1494.1565 1521.1633]

Cluster 5 (3 m/z features):
[604.5345 775.5356 825.5629]

Cluster 6 (15 m/z features):
[371.9835 460.0332 507.0164 511.0158 546.9926 547.9938 571.9841 681.0148
 682.0179 687.021

In [90]:
clustered_mzs = df_metrics_clean.groupby("cluster")["mz"].apply(list).to_dict()


In [91]:
df_metrics_clean[["mz", "cluster"]].sort_values("cluster").to_csv("clustered_mzs.csv", index=False)


In [92]:
cluster_counts = df_metrics_clean["cluster"].value_counts().sort_index()
print(cluster_counts)

cluster
0      9
1     21
2      6
3     21
4     14
      ..
95    12
96     2
97    13
98    16
99     2
Name: count, Length: 100, dtype: int64


In [93]:
import os
import plotly.io as pio

# Ensure the image folder exists
os.makedirs(f"images_from_clusters_adducts", exist_ok=True)

# Merge cluster info with AnnData's m/z values
mz_cluster_map = dict(zip(df_metrics_clean["mz"].astype(float), df_metrics_clean["cluster"]))

# Compute the global spectrum
global_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()

# Get indices of m/z values sorted by intensity (descending)
sorted_indices = np.argsort(global_spectrum)[::-1]

# Select indices of interest (e.g., all, top N, etc.)
selected_indices = sorted_indices[0:]  # You can limit with [10:20] if desired

# Get corresponding m/z values from adata.var
selected_mz = adata.var["mzs"].values[selected_indices].astype(float)

# Generate and save images
for mz in selected_mz:
    if mz not in mz_cluster_map:
        print(f"Skipping m/z {mz:.4f} — not in clustered data.")
        continue

    cluster_id = mz_cluster_map[mz]
    fig = plot_mz_across_sample_custom_tic(adata, mz, tol=0.001)  # Make sure this function returns a Plotly fig

    filename = f"images_from_clusters_adducts/cluster_{cluster_id}_mz_{mz:.4f}_.png"
    pio.write_image(fig, filename)
    print(f"Saved: {filename}")

Saved: images_from_clusters_adducts/cluster_56_mz_273.0397_.png
Saved: images_from_clusters_adducts/cluster_56_mz_137.0251_.png
Saved: images_from_clusters_adducts/cluster_56_mz_274.0435_.png
Saved: images_from_clusters_adducts/cluster_23_mz_369.3523_.png
Saved: images_from_clusters_adducts/cluster_84_mz_331.0297_.png
Saved: images_from_clusters_adducts/cluster_23_mz_257.0478_.png
Saved: images_from_clusters_adducts/cluster_49_mz_195.0905_.png
Saved: images_from_clusters_adducts/cluster_84_mz_637.3020_.png
Saved: images_from_clusters_adducts/cluster_43_mz_313.0337_.png
Saved: images_from_clusters_adducts/cluster_56_mz_275.0563_.png
Saved: images_from_clusters_adducts/cluster_43_mz_245.0458_.png
Saved: images_from_clusters_adducts/cluster_84_mz_333.0266_.png
Saved: images_from_clusters_adducts/cluster_80_mz_197.9919_.png
Saved: images_from_clusters_adducts/cluster_56_mz_244.0353_.png
Saved: images_from_clusters_adducts/cluster_56_mz_246.0516_.png
Saved: images_from_clusters_adducts/clus

Processing m/z features: 100%|██████████| 1000/1000 [19:55<00:00,  1.20s/it]
/opt/anaconda3/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



         mz  moran_I_tissue  geary_C_tissue  entropy_tissue  cv_tissue  \
0  136.0150        0.415658        0.547822        8.402684   0.965441   
1  136.0608        0.262929        0.729385        8.702899   0.630088   
2  137.0251        0.549667        0.406121        8.616154   0.964382   
3  138.0287        0.508409        0.452354        8.596314   0.921969   
4  139.0359        0.363029        0.612376        8.537266   0.820836   

   skew_tissue  region_size  moran_I_background  geary_C_background  \
0     3.751760         6199            0.493544            0.501015   
1     2.034254         7144            0.494965            0.504816   
2     7.986627         7151            0.721676            0.275891   
3     5.827630         7122            0.701255            0.295633   
4     3.310123         6636            0.650540            0.345992   

   entropy_background  ...  has_H_adduct  has_Na_adduct  has_K_adduct  \
0            9.055275  ...             1              1

(array([-1,  0,  1,  2,  3]), array([598,  63, 122,  36, 181]))
